# Приём показаний ESP32

DHT22 передаёт температуру и влажность через ESP32 по USB. Python сохраняет их в SQLite и проверяет пороги. Перед запуском закрой монитор порта Arduino IDE. База должна быть подготовлена в первом блокноте.

## 1. Настройки и подключение
Укажи порт своей платы. Скорость 115200 совпадает с прошивкой. Один запуск сохраняет пять пар показаний. Если корректные данные не приходят 15 секунд, сбор останавливается.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sqlite3
import serial
import time
import math

port = "/dev/cu.usbserial-10"
baudrate = 115200
pair_count = 5
idle_timeout = 15

db_path = Path("microclimate.db").resolve()
if not db_path.is_file():
    raise FileNotFoundError("База не найдена. Сначала выполни первый блокнот.")
if "conn" in globals():
    conn.close()
conn = sqlite3.connect(f"{db_path.as_uri()}?mode=rw", uri=True)
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")

rows = conn.execute("""
    SELECT channel.id, channel.metric
    FROM channel
    INNER JOIN sensor ON sensor.id = channel.sensor_id
    INNER JOIN node ON node.id = sensor.node_id
    WHERE node.device_uid = ? AND sensor.model = ?
""", ("ESP32-001", "DHT22")).fetchall()

channels = {row["metric"]: row["id"] for row in rows}
if len(rows) != 2 or set(channels) != {"temperature", "humidity"}:
    raise ValueError("Для узла нужны два канала одного DHT22.")

print("База:", db_path)
print("Каналы:", channels)

## 2. Проверка порогов
Выбираем правило, действовавшее в момент получения показаний. Значения на самой границе допустимы. Ниже границы — below, выше — above. Если подходящего правила нет, измерение сохраняется без предупреждения. Пересечение правил считается ошибкой.

In [ ]:
def check_threshold(conn, channel_id, ts, value):
    rule = conn.execute("""
        SELECT id, lower_bound, upper_bound
        FROM threshold
        WHERE channel_id = ?
          AND valid_from <= ?
          AND (valid_to IS NULL OR ? < valid_to)
    """, (channel_id, ts, ts)).fetchall()

    if len(rule) == 0:
        return "Нет действующего порога"

    if len(rule) > 1:
        raise ValueError("Для канала действуют несколько порогов одновременно.")

    threshold_id, lower, upper = rule[0]

    if value < lower:
        direction = "below"
    elif value > upper:
        direction = "above"
    else:
        return "В пределах порога"

    conn.execute("""
        INSERT INTO alert (threshold_id, channel_id, ts, direction)
        VALUES (?, ?, ?, ?)
        ON CONFLICT (channel_id, ts) DO NOTHING
    """, (threshold_id, channel_id, ts, direction))

    return direction

## 3. Получение и сохранение
Пустые, служебные и повреждённые строки пропускаются. Время получения отмечается на компьютере в UTC. Два измерения и их предупреждения сохраняются одной транзакцией. При остановке уже сохранённые пары остаются в базе. Эту ячейку можно запускать повторно для новых показаний.

In [ ]:
try:
    saved = 0
    last_received = time.monotonic()

    with serial.Serial(port, baudrate, timeout=1) as esp:
        while saved < pair_count:
            if time.monotonic() - last_received >= idle_timeout:
                raise TimeoutError(f"Нет корректных показаний уже {idle_timeout} секунд.")

            sample = esp.readline().decode("utf-8", errors="replace").strip()
            parts = sample.split()

            if len(parts) != 7:
                continue

            if (
                parts[0] != "Temperature:"
                or parts[2] != "C"
                or parts[3] != "|"
                or parts[4] != "Humidity:"
                or parts[6] != "%"
            ):
                continue

            try:
                temperature = float(parts[1])
                humidity = float(parts[5])
            except ValueError:
                continue

            if not (-40 <= temperature <= 80 and 0 <= humidity <= 100):
                continue

            last_received = time.monotonic()
            ts = datetime.now(timezone.utc).isoformat(
                timespec="milliseconds"
            ).replace("+00:00", "Z")

            with conn:
                conn.execute(
                    "INSERT INTO measurement (channel_id, ts, value) VALUES (?, ?, ?)",
                    (channels["temperature"], ts, temperature)
                )
                conn.execute(
                    "INSERT INTO measurement (channel_id, ts, value) VALUES (?, ?, ?)",
                    (channels["humidity"], ts, humidity)
                )

                temperature_result = check_threshold(conn, channels["temperature"], ts, temperature)
                humidity_result = check_threshold(conn, channels["humidity"], ts, humidity)

            saved += 1
            print(
                "Сохранена пара:", saved,
                "| Температура:", temperature,
                "| Влажность:", humidity,
                "|", temperature_result, "|", humidity_result
            )
except KeyboardInterrupt:
    print("Сбор остановлен пользователем. Сохранено пар:", saved)
except (serial.SerialException, OSError, TimeoutError, ValueError,
        sqlite3.IntegrityError, sqlite3.OperationalError) as error:
    print("Сбор остановлен:", error)

## 4. Последние измерения и предупреждения
Эту ячейку можно выполнить без платы, пропустив сбор.

In [ ]:
rows = conn.execute("""
    SELECT channel.metric, measurement.ts, measurement.value, channel.unit
    FROM measurement
    INNER JOIN channel ON channel.id = measurement.channel_id
    ORDER BY measurement.ts DESC, channel.id
    LIMIT 10
""").fetchall()

print("Последние измерения (UTC):")
for row in rows:
    print(dict(row))

alerts = conn.execute("""
    SELECT channel.metric, alert.ts, measurement.value,
           threshold.lower_bound, threshold.upper_bound, alert.direction
    FROM alert
    INNER JOIN channel ON channel.id = alert.channel_id
    INNER JOIN measurement
        ON measurement.channel_id = alert.channel_id AND measurement.ts = alert.ts
    INNER JOIN threshold ON threshold.id = alert.threshold_id
    ORDER BY alert.ts DESC, channel.id
    LIMIT 10
""").fetchall()

print("\nПоследние предупреждения:")
if not alerts:
    print("Предупреждений нет.")
for row in alerts:
    print(dict(row))

## 5. Изменение границ
Эта ячейка только определяет функцию. При необходимости вызови, например, `set_threshold(conn, channels["temperature"], 20.0, 27.0)` перед закрытием соединения. Новое правило начнёт действовать с текущего времени. Старые границы и предупреждения сохранятся.

In [ ]:
def set_threshold(conn, channel_id, lower, upper, valid_from=None):
    """Добавляет новое правило, завершая предыдущее бессрочное.

    Изменения только вперёд по времени: старые измерения и alert не переписываются.
    Периоды имеют вид [valid_from, valid_to).
    """
    if not (math.isfinite(lower) and math.isfinite(upper) and lower < upper):
        raise ValueError("Границы должны быть конечными числами, нижняя меньше верхней.")
    start = (datetime.now(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")
             if valid_from is None else valid_from)
    parsed = datetime.fromisoformat(start.replace("Z", "+00:00"))
    if parsed.tzinfo is None or parsed.utcoffset().total_seconds() != 0:
        raise ValueError("Начало действия должно быть в UTC.")
    canonical = parsed.isoformat(timespec="milliseconds").replace("+00:00", "Z")
    if start != canonical:
        raise ValueError("Ожидается UTC-формат YYYY-MM-DDTHH:MM:SS.sssZ.")
    with conn:
        if conn.execute("SELECT id FROM channel WHERE id = ?", (channel_id,)).fetchone() is None:
            raise ValueError("Канал не найден.")
        rules = conn.execute("""
            SELECT id, valid_from, valid_to FROM threshold
            WHERE channel_id = ? ORDER BY valid_from
        """, (channel_id,)).fetchall()
        for rule in rules:
            if rule["valid_from"] >= start:
                raise ValueError("Новое правило должно начинаться позже всех существующих.")
            if rule["valid_to"] is not None and rule["valid_to"] > start:
                raise ValueError("Новое правило пересекает завершённый период.")
        open_rules = [rule for rule in rules if rule["valid_to"] is None]
        if len(open_rules) > 1:
            raise ValueError("Найдены пересекающиеся бессрочные правила.")
        if open_rules:
            conn.execute("UPDATE threshold SET valid_to = ? WHERE id = ?", (start, open_rules[0]["id"]))
        cursor = conn.execute("""
            INSERT INTO threshold (channel_id, lower_bound, upper_bound, valid_from, valid_to)
            VALUES (?, ?, ?, ?, NULL)
        """, (channel_id, lower, upper, start))
        return cursor.lastrowid

## 6. Завершение
Порт закрывается автоматически после сбора. Соединение с базой закрываем здесь. Для нового сеанса начни с первой ячейки.

In [ ]:
conn.close()
print("Соединение закрыто.")